# Phase B — Data Preparation Pipeline

This is the **Data Preparation** phase of CRISP-DM. It turns the raw sales data into modeling-ready,
leakage-safe datasets, applying the decisions justified in **Phase A** (`Phase_A_Evidence.ipynb`):

- **Hit label** (Part 2): year-based top-25% sales OR critic_score >= 8.5.
- **Features** (Parts 1 & 3): Genre_Clean, Platform_Family, and the train-only Publisher_Tier.
- **Scope variants** (Parts 1 & 4): no-cutoff / 2000-single / 3 data-driven eras.

The hard rules are enforced here: **split first**, fit every distribution-based feature (Publisher_Tier)
on the **train set only**, drop all leakage columns, and use `random_state=42` throughout. The output
(`data/processed/<scope>/`) is consumed by **Phase C** (`Phase_C_Modeling.ipynb`).

## Setup

Importing libraries and setting global parameters.

In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split

# Set random state for reproducibility
RANDOM_STATE = 42

# Create directory for processed data if it doesn't exist
os.makedirs('data/processed', exist_ok=True)

print('Setup complete.')

Setup complete.


## Data Loading and Initial Cleaning

We load the raw dataset and filter for 'usable' rows. A row is usable if it has either a sales figure or a critic score. We also parse the release date to extract the year, as our target definition relies on year-based sales percentiles.

In [2]:
# Load raw data
raw_df = pd.read_csv('data/Video Games Sales (1980-2024) - Raw.csv')

# Parse year
raw_df['year'] = pd.to_datetime(raw_df['release_date'], errors='coerce', dayfirst=True).dt.year

# Filter for usable rows and non-null year
# Usable: total_sales is not null OR critic_score is not null
df_usable = raw_df[(raw_df['total_sales'].notna() | raw_df['critic_score'].notna()) & raw_df['year'].notna()].copy()

print(f"Raw rows: {len(raw_df)}")
print(f"Usable rows (with year): {len(df_usable)}")

Raw rows: 64016
Usable rows (with year): 21343


## Step 1: Hit Label Definition

**Definition:** A game is a 'Hit' if it meets either a commercial threshold (Top 25% of sales in its release year) or a critical threshold (Critic Score of 8.5 or higher).

**Leakage Note:** The 'Hit' label is our target variable. While it depends on 'future' data (sales and scores), it is the ground truth we aim to predict using *only* pre-release features. Thus, computing it on the full usable dataset is permissible and necessary for a stratified split.

In [3]:
# Calculate 75th percentile of total_sales per year
annual_thresholds = df_usable[df_usable['total_sales'].notna()].groupby('year')['total_sales'].quantile(0.75)

# Function to determine if it's a sales hit
def is_sales_hit(row):
    if pd.isna(row['total_sales']):
        return False
    return row['total_sales'] >= annual_thresholds[row['year']]

df_usable['sales_hit'] = df_usable.apply(is_sales_hit, axis=1)
df_usable['score_hit'] = df_usable['critic_score'] >= 8.5

# Final Hit label
df_usable['Hit'] = ((df_usable['sales_hit'] == True) | (df_usable['score_hit'] == True)).astype(int)

# Drop intermediate helper cols
df_usable = df_usable.drop(columns=['sales_hit', 'score_hit'])

print(f"Overall Hit rate: {df_usable['Hit'].mean():.2%}")
print("Class balance:")
print(df_usable['Hit'].value_counts(normalize=True))

Overall Hit rate: 26.32%
Class balance:
Hit
0    0.736822
1    0.263178
Name: proportion, dtype: float64


## Step 2: Feature Engineering (Deterministic)

We create deterministic features that do not depend on the data distribution (and thus don't cause leakage if computed before the split).

1. **Genre_Clean**: We consolidate low-count genres (e.g., Board Game, Sandbox) into an 'Other' category to reduce dimensionality and improve model robustness.
2. **Platform_Family**: We group individual consoles into broad families (Playstation, Xbox, Nintendo, PC, Mobile).

In [4]:
def clean_features(df):
    df = df.copy()
    
    # Genre Clean
    small_genres = ['Board Game','Sandbox','Education','Party','MMO','Music','Visual Novel']
    df['Genre_Clean'] = df['genre'].apply(lambda x: 'Other' if x in small_genres else x)
    
    # Platform Family
    playstation = ['PS','PS2','PS3','PS4','PS5','PSP','PSV','PSN','BRW']
    xbox = ['XB','X360','XOne','XS','XBL']  # 'Series' removed: it is a generic catalog label (no sales/score, dropped as unusable), not Xbox Series X/S (='XS')
    nintendo = ['DS','GBA','Wii','NS','3DS','WiiU','GC','N64','GB','DSi','DSiW','VC','GBC','WW','WS']
    pc = ['PC','OSX','Linux','WinP']
    mobile = ['And','iOS','Mob']
    
    def get_family(console):
        if console in playstation: return 'Playstation'
        if console in xbox: return 'Xbox'
        if console in nintendo: return 'Nintendo'
        if console in pc: return 'PC'
        if console in mobile: return 'Mobile'
        return 'Other'
        
    df['Platform_Family'] = df['console'].apply(get_family)
    return df

df_usable = clean_features(df_usable)


## Step 3: The Leakage-Safe prepare() Routine

This function encapsulates the core data preparation logic, strictly enforcing leakage prevention rules:
1. **Stratified Split FIRST**: No distribution-based feature (like Publisher Tier) is calculated before the split.
2. **Publisher Tier on TRAIN only**: We determine publisher reputation based *only* on the training set. The test set uses a lookup, falling back to 'Tier 1' for unknown publishers.
3. **Feature Selection**: We drop all 'future' data (sales, scores) and raw identifiers, keeping only the prepared categorical features.

In [5]:
def prepare(df_subset, name):
    print(f"--- Preparing: {name} ---")
    
    # a) Stratified split
    train_df, test_df = train_test_split(
        df_subset, 
        test_size=0.2, 
        random_state=RANDOM_STATE, 
        stratify=df_subset['Hit']
    )
    
    # b) Publisher_Tier on TRAIN ONLY
    # Dedup and aggregate
    pub_stats = train_df.groupby(['title', 'publisher']).agg({
        'total_sales': 'sum',
        'critic_score': 'max'
    }).reset_index()
    
    # Define iconic games
    pub_stats['iconic'] = (pub_stats['total_sales'] > 10) | (pub_stats['critic_score'] >= 9)
    
    # Count iconic games per publisher
    iconic_counts = pub_stats[pub_stats['iconic']].groupby('publisher').size()
    
    def get_tier(count):
        if count >= 5: return 'Tier3'
        if count >= 1: return 'Tier2'
        return 'Tier1'
        
    pub_tier_map = iconic_counts.apply(get_tier).to_dict()
    
    # Map onto TRAIN
    train_df['Publisher_Tier'] = train_df['publisher'].map(pub_tier_map).fillna('Tier1')
    
    # Map onto TEST (Lookup, unseen -> Tier1)
    test_df['Publisher_Tier'] = test_df['publisher'].map(pub_tier_map).fillna('Tier1')
    
    # c) Select features
    features = ['Genre_Clean', 'Platform_Family', 'Publisher_Tier']
    target = 'Hit'
    
    X_train = train_df[features]
    y_train = train_df[target]
    X_test = test_df[features]
    y_test = test_df[target]
    
    # Sanity checks
    print(f"Rows: Train={len(X_train)}, Test={len(X_test)}")
    print(f"Hit %: Train={y_train.mean():.1%}, Test={y_test.mean():.1%}")
    print("Train Tier distribution:")
    print(X_train['Publisher_Tier'].value_counts())
    
    unseen_count = test_df[~test_df['publisher'].isin(pub_tier_map.keys())].shape[0]
    print(f"Test rows with unseen publishers (fallback to Tier1): {unseen_count}")
    
    return X_train, X_test, y_train, y_test

## Step 4: Building Scope Variants

We prepare three scope variants to test the impact of historical data and market segmentation on model performance:
1. **Scope 1 (No Cutoff)**: All data up to 2018.
2. **Scope 2 (2000 Single)**: Focus on the 'modern' era (2000-2018).
3. **Scope 3 (Eras)**: Segmented models for three distinct market eras (2000-06, 2007-11, 2012-18).

In [6]:
# Data capped at 2018
df_cap = df_usable[df_usable['year'] <= 2018].copy()

# Scope 1: No Cutoff
results = {}
results['scope1_nocutoff'] = prepare(df_cap, "Scope 1 (Full <= 2018)")

# Scope 2: 2000 Single
df_2000 = df_cap[df_cap['year'] >= 2000].copy()
results['scope2_2000single'] = prepare(df_2000, "Scope 2 (2000-2018)")

# Scope 3: Eras
era_definitions = {
    'era1_2000-2006': (2000, 2006),
    'era2_2007-2011': (2007, 2011),
    'era3_2012-2018': (2012, 2018)
}

for era_name, (start, end) in era_definitions.items():
    era_df = df_cap[(df_cap['year'] >= start) & (df_cap['year'] <= end)].copy()
    results[era_name] = prepare(era_df, f"Scope 3 - {era_name}")

--- Preparing: Scope 1 (Full <= 2018) ---
Rows: Train=16992, Test=4248
Hit %: Train=26.3%, Test=26.3%
Train Tier distribution:
Publisher_Tier
Tier3    6975
Tier1    5462
Tier2    4555
Name: count, dtype: int64
Test rows with unseen publishers (fallback to Tier1): 1407
--- Preparing: Scope 2 (2000-2018) ---
Rows: Train=15472, Test=3868
Hit %: Train=26.0%, Test=26.0%
Train Tier distribution:
Publisher_Tier
Tier3    6930
Tier1    5268
Tier2    3274
Name: count, dtype: int64
Test rows with unseen publishers (fallback to Tier1): 1413
--- Preparing: Scope 3 - era1_2000-2006 ---
Rows: Train=4876, Test=1219
Hit %: Train=26.7%, Test=26.7%
Train Tier distribution:
Publisher_Tier
Tier3    1794
Tier1    1664
Tier2    1418
Name: count, dtype: int64
Test rows with unseen publishers (fallback to Tier1): 405
--- Preparing: Scope 3 - era2_2007-2011 ---
Rows: Train=6854, Test=1714
Hit %: Train=25.0%, Test=25.0%
Train Tier distribution:
Publisher_Tier
Tier1    2746
Tier2    2485
Tier3    1623
Name: count

## Step 5: Save Processed Data

Saving the prepared datasets into structured directories for Phase C (Modeling).

In [7]:
def save_results(name, data_tuple):
    X_train, X_test, y_train, y_test = data_tuple
    
    if 'era' in name:
        base_path = f'data/processed/scope3_eras/{name}'
    else:
        base_path = f'data/processed/{name}'
        
    os.makedirs(base_path, exist_ok=True)
    
    X_train.to_csv(f'{base_path}/X_train.csv', index=False)
    X_test.to_csv(f'{base_path}/X_test.csv', index=False)
    y_train.to_csv(f'{base_path}/y_train.csv', index=False)
    y_test.to_csv(f'{base_path}/y_test.csv', index=False)
    print(f"Saved {name} to {base_path}")

for name, data in results.items():
    save_results(name, data)

Saved scope1_nocutoff to data/processed/scope1_nocutoff


Saved scope2_2000single to data/processed/scope2_2000single
Saved era1_2000-2006 to data/processed/scope3_eras/era1_2000-2006
Saved era2_2007-2011 to data/processed/scope3_eras/era2_2007-2011
Saved era3_2012-2018 to data/processed/scope3_eras/era3_2012-2018


## Step 6: Final Summary

Reviewing the consistency of our prepared datasets.

In [8]:
summary_list = []
for name, (X_train, X_test, y_train, y_test) in results.items():
    summary_list.append({
        'Scope/Era': name,
        'N Train': len(X_train),
        'N Test': len(X_test),
        'Train Hit %': f"{y_train.mean():.1%}",
        'Test Hit %': f"{y_test.mean():.1%}",
        'Tier 1/2/3 (Train)': f"{list(X_train['Publisher_Tier'].value_counts().sort_index().values)}"
    })

summary_df = pd.DataFrame(summary_list)
print(summary_df.to_string(index=False))

# Final check for leakage columns
forbidden = ['total_sales', 'critic_score', 'na_sales', 'jp_sales', 'pal_sales', 'other_sales', 'iconic']
for name, (X_train, _, _, _) in results.items():
    found = [c for c in forbidden if c in X_train.columns]
    if found:
        print(f"CRITICAL ERROR: Leakage columns found in {name}: {found}")
    else:
        print(f"Verified {name}: No leakage columns.")

print('\nPHASE B DONE')

        Scope/Era  N Train  N Test Train Hit % Test Hit %                               Tier 1/2/3 (Train)
  scope1_nocutoff    16992    4248       26.3%      26.3% [np.int64(5462), np.int64(4555), np.int64(6975)]
scope2_2000single    15472    3868       26.0%      26.0% [np.int64(5268), np.int64(3274), np.int64(6930)]
   era1_2000-2006     4876    1219       26.7%      26.7% [np.int64(1664), np.int64(1418), np.int64(1794)]
   era2_2007-2011     6854    1714       25.0%      25.0% [np.int64(2746), np.int64(2485), np.int64(1623)]
   era3_2012-2018     3741     936       26.7%      26.7%   [np.int64(2350), np.int64(722), np.int64(669)]
Verified scope1_nocutoff: No leakage columns.
Verified scope2_2000single: No leakage columns.
Verified era1_2000-2006: No leakage columns.
Verified era2_2007-2011: No leakage columns.
Verified era3_2012-2018: No leakage columns.

PHASE B DONE
